# CorpPolicyLM — Assignment 1A: Continual Pre-Training + QLoRA Instruction Fine-Tuning

**Domain:** HR Policy & Corporate Documents (Enterprise Variant 1 — Domain Q&A Assistant)
**Model:** `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T` (HR row, T4 — Choice 2) **GPU:** Colab T4 (16 GB)
**Pipeline:** raw policy PDFs → cleaned corpus → packed tokens → CPT → perplexity & forgetting → instruction dataset → QLoRA (Adapter B) → evaluation

| Part | Step | Marks |
|---|---|---|
| A | 1 Data collection, extraction & cleaning | 2 |
| A | 2 Tokenisation & packed dataset | 2 |
| A | 3 Model loading & architecture inspection | 2 |
| A | 4 CPT training loop & loss analysis | 2 |
| A | 5 Perplexity & catastrophic forgetting | 2 |
| B | B1 Instruction dataset · B2 QLoRA (Adapter B) · B3 Evaluation | 2 + 2 + 1 |

### Why TinyLlama-1.1B?
* It is the HR-Policy model listed for a T4 GPU, and at 1.1 B parameters a *full-parameter* CPT fits in 16 GB with gradient checkpointing + 8-bit paged AdamW.
* It is a LLaMA-architecture model, so its attention projections are literally named `q_proj`, `k_proj`, `v_proj`, `o_proj` — the target modules in the QLoRA adapter table map one-to-one.
* It was pre-trained on ~3 T tokens of general web/code text, so it knows general English but has seen little Indian HR-policy language (band levels, EL/CL/SL, notice periods, POSH/ICC, vigil mechanisms) — a realistic domain gap for CPT to close.
* The *base* checkpoint (not the chat model) is used for CPT, and its tokenizer is used for every step — no tokenizer swapping.

All logic lives in the `src/` package (one module per step); this notebook calls it, shows the outputs, and records inferences.

## 0. Environment setup
Upload the `CorpPolicyLM` project folder to Google Drive (`MyDrive/CorpPolicyLM`) and run this notebook on a **T4 GPU** runtime. Checkpoints are written to Drive so they survive a runtime reset.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CorpPolicyLM"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("Project:", PROJECT_DIR)

In [ ]:
%pip install -q -r requirements.txt

> **First run on a fresh Colab runtime:** the cell above upgrades `transformers`/`trl`/`peft`. Use *Runtime → Restart session* once, then run the setup cell again and continue from here.

In [ ]:
import json, torch, pandas as pd
from IPython.display import display, Image, Markdown
pd.set_option("display.max_colwidth", 200)
from src import config
config.ensure_dirs()
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB | compute capability {p.major}.{p.minor}")
print("Model:", config.MODEL_ID, "| block size:", config.BLOCK_SIZE)

# PART A — Continual Pre-Training (CPT)

## Step 1 — Data collection, extraction & cleaning  [2 marks]

### 1a. Data collection
The starting corpus was 4 MyGov (Govt. of India) HR policies — *Leave, Attendance & Work Protocols, Exit, Domestic Travel* (~4.3 K words ≈ 6 K tokens). That is only 2–3 packed 2048-token sequences: far too little for a meaningful loss curve or a 10 % held-out set. It was expanded with public HR / corporate-policy PDFs listed in `data/sources.csv`:

| Category | Examples |
|---|---|
| Government HR rules | CCS (Leave) Rules 1972, CCS (Conduct) Rules 1964, MyGov laptop / talent-development / internship policies |
| Organisation HR manuals | IIM Udaipur (permanent, contractual & faculty), IIM Raipur, IIHMR, AICB, IILM, Indus & DAV universities, INOXCVA, MSRLS |
| Leave rules | NIT Tiruchirappalli, Shiv Nadar University |
| POSH policies | BASF, 3M, Sony Pictures Networks, Bajaj Broking, JioStar, Tata Elxsi, BPTP, Sudarshan, Mafatlal |
| Code of conduct / anti-bribery / human rights | Bharat Forge, Accelya, Niramai, EID Parry, Aurobindo, Oil India, Infosys, Home First |
| Whistle-blower / vigil mechanism | AU Bank, BEML, IKS Health, Astral, GHL, Khanna Paper, DreamFolks, GrowXCD |

One source (`mygov_travel_policy_dup.pdf`) is deliberately the same file as `travel_policy.pdf` to verify that the deduplication stage works. `download_corpus` is idempotent: files already present are skipped, so re-running only fetches what is missing. It sends browser-like headers (several institutional sites reject scripted user agents with 403/404), retries transient errors, and verifies TLS with the `certifi` CA bundle. URLs that are permanently dead are logged in `download_report.csv` and never stop the run.

In [ ]:
from src.download_corpus import download_corpus
dl = pd.DataFrame(download_corpus())
display(dl[["file_name", "category", "status", "bytes"]])
print(dl["status"].str.split(":").str[0].value_counts().to_string())

### 1b. Page-by-page extraction (`src/extract.py`, pypdf)
Each PDF is read page by page; every page is written with a `--- PAGE n ---` marker so the cleaner can detect running headers/footers. Pages with no extractable text (scanned images) are counted.

In [ ]:
from src.extract import extract_corpus
ext = pd.DataFrame(extract_corpus(config.RAW_PDF_DIR, config.EXTRACTED_TEXT_DIR,
                                  config.REPORT_DIR / "extraction_report.csv"))
display(ext.sort_values("total_pages", ascending=False))
print(f"{len(ext)} PDFs | {ext.total_pages.sum()} pages | {ext.empty_pages.sum()} pages without text | "
      f"{ext.characters_extracted.sum():,} characters")

### 1c. Cleaning pipeline (`src/clean.py`)
| Stage | What it does | Why |
|---|---|---|
| 0 basic cleaning | NFKC normalisation, control-character removal, de-hyphenation, drop page markers / page numbers, drop lines repeated on ≥ 50 % of pages (headers, footers, document-control blocks), collapse whitespace | Boilerplate repeated on every page would be over-learned by CPT and inflates token counts |
| 1 length filter | drop documents < 100 words | near-empty extractions (scanned PDFs, cover pages) carry no language signal |
| 2 exact dedup | SHA-256 of whitespace/case-normalised text | identical files would be seen twice per epoch → memorisation |
| 3 near dedup | 5-word-shingle Jaccard ≥ 0.8 against kept docs | revised versions of the same policy (e.g. two INOXCVA manuals) are ~90 % identical |
| 4 language filter | `langdetect` (seeded) keeps `en` only | bilingual Hindi/English government PDFs would add a second language |

In [ ]:
from src.clean import clean_corpus, summarize_corpus
records = clean_corpus(config.EXTRACTED_TEXT_DIR, config.DOMAIN_CORPUS_DIR, config.REPORT_DIR / "cleaning_report.csv")
summary = summarize_corpus(records)
(config.REPORT_DIR / "corpus_summary.json").write_text(json.dumps(summary, indent=2))
stages = pd.DataFrame(summary["stages"])
display(stages)
rej = pd.DataFrame(records).query("status == 'rejected'")
display(rej[["file_name", "reason", "duplicate_of", "max_similarity", "language", "word_count"]])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(stages.stage, stages.documents_after, color="#1f4e8c"); ax[0].set_title("Documents remaining after each stage")
ax[1].bar(stages.stage, stages.characters_removed, color="#c0504d"); ax[1].set_title("Characters removed by each stage")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(config.FIGURE_DIR / "cleaning_stages.png", dpi=150); plt.show()

w = summary["retained_document_words"]
biggest_docs = summary["largest_document_removal_stages"] or ["none"]
biggest_chars = summary["largest_character_removal_stages"] or ["none"]
print(f"Documents: {summary['documents_before']} → {summary['documents_after']} | characters: "
      f"{summary['characters_before']:,} → {summary['characters_after']:,} (−{summary['character_reduction_percent']}%)")
print(f"Greatest impact on document count : {', '.join(biggest_docs)}")
print(f"Greatest impact on corpus size (chars): {', '.join(biggest_chars)}")
print(f"Retained words: total {w['total']:,} | per doc min {w['minimum']} / median {w['median']} / max {w['maximum']}")

**Inferences — Step 1**
* *Which step had the greatest impact?* Read the two lines printed above. By **document count** the de-duplication stages (exact + near) remove the most files — the planted duplicate travel policy, and revised editions of the same HR manual / boiler-plate vigil-mechanism policies that companies copy from the SEBI LODR template. By **corpus size** the biggest cut usually comes from removing a *long* near-duplicate manual, while basic cleaning removes a steady ~5–8 % of characters (page numbers, running headers, document-control blocks) from every document.
* Length filtering mainly catches scanned PDFs whose pages have no text layer (`empty_pages` in the extraction table) — those would otherwise enter CPT as empty documents.
* The language filter rarely fires on this corpus because most Indian HR policies are English-only; it guards against bilingual government circulars.
* Exact-hash dedup alone would miss *revised* versions of a policy; the shingle-Jaccard stage catches them (see `max_similarity` in the rejected table). Without it, CPT would see the same clauses twice per epoch and the held-out PPL could be contaminated if one edition lands in train and the other in eval.

## Step 2 — Tokenisation & packed dataset  [2 marks]

1. **Held-out split first (`src/dataset.py`)** — 10 % of the cleaned *documents* (seed 42) are reserved for Step 5A *before* tokenisation, so no held-out sentence can leak into a training block.
2. **Tokenizer** — `AutoTokenizer.from_pretrained(MODEL_ID)`; the pretrained LLaMA SentencePiece vocabulary (32 000) is reused, never retrained, so every token id still maps to the embedding row the model learned.
3. **BOS/EOS** — each document becomes `<s> … </s>` so the model sees document boundaries inside a packed block.
4. **Packing** — all wrapped documents are concatenated into one stream and cut into 2048-token blocks (TinyLlama's context window): zero padding, 100 % of every forward pass is real text. The incomplete tail of the training stream is dropped; the held-out tail is kept so PPL covers every eval token.
5. Saved as Parquet (`data/processed/{train,eval}_packed.parquet`).

In [ ]:
from src.dataset import split_corpus
split = split_corpus()
print(split["note"])
from src.tokenize_pack import build_packed_dataset
tok_stats = build_packed_dataset()
display(pd.DataFrame(tok_stats["splits"]).T)
print({k: v for k, v in tok_stats.items() if k != "splits"})

In [ ]:
from src.tokenize_pack import load_tokenizer
tokenizer = load_tokenizer()
train_blocks = pd.read_parquet(config.PROCESSED_DIR / "train_packed.parquet")
first = list(train_blocks.input_ids.iloc[0])
print("Block 0 starts with:", tokenizer.convert_ids_to_tokens(first[:12]))
pos = [i for i, t in enumerate(first) if t == tokenizer.eos_token_id][:1]
if pos:
    i = pos[0]; print("Document boundary inside block 0:", tokenizer.decode(first[i-15:i+8]))
doc_tokens = pd.read_csv(config.PROCESSED_DIR / "train_doc_tokens.csv")
doc_tokens.tokens.plot.hist(bins=30, title="Tokens per training document", figsize=(7, 3)); plt.show()
s = tok_stats["splits"]
print(f"Total tokens: {s['train']['total_tokens'] + s['eval']['total_tokens']:,} | avg document length: "
      f"{(s['train']['total_tokens'] + s['eval']['total_tokens']) / (s['train']['documents'] + s['eval']['documents']):,.0f} tokens | "
      f"packed sequences: {s['train']['packed_sequences']} train + {s['eval']['packed_sequences']} eval")

**Inferences — Step 2**
* Words→tokens ratio is roughly 1.3–1.4 tokens/word: HR text is full of abbreviations (EL, CL, ICC, DA, TA), band codes (E1, J2, M3) and rupee amounts (`Rs.10`, `3500/-`) that SentencePiece splits into several pieces — exactly the domain-specific patterns CPT should make cheaper to predict.
* Document lengths are highly skewed (2-page POSH policies vs. 100-page HR manuals), so without packing most batches would be dominated by padding; packing gives fixed-size, padding-free blocks and full GPU utilisation.
* Because documents are much longer or much shorter than 2048 tokens, many blocks contain an `</s><s>` boundary; attention is not reset across it, which is the standard trade-off of packed CPT (the BOS/EOS markers teach the model that context before `</s>` belongs to a different document).
* Only the final `< 2048` tokens of the training stream are discarded (< 0.1 % of the corpus).

## Step 3 — Model loading & architecture inspection  [2 marks]
Loaded with `AutoModelForCausalLM.from_pretrained(..., dtype=torch.bfloat16)` and gradient checkpointing enabled, as the assignment specifies for T4.

In [ ]:
from src.model_utils import load_model, architecture_audit, generate, save_json
model = load_model(config.MODEL_ID, dtype=torch.bfloat16, gradient_checkpointing=True)
audit = architecture_audit(model, tokenizer)
save_json(audit, config.EVAL_DIR / "architecture_audit.json")
display(pd.Series(audit, name="value").to_frame())
print(model)

In [ ]:
print(f"Trainable parameters: {audit['trainable_parameters']:,} ({audit['trainable_parameters']/1e9:.2f} B)")
print(f"Decoder layers {audit['num_decoder_layers']} | attention heads {audit['num_attention_heads']} "
      f"(KV heads {audit['num_key_value_heads']}) | hidden {audit['hidden_size']} | head dim {audit['head_dim']}")
print(f"lm_head {audit['lm_head_shape']} -> output dim {audit['lm_head_shape'][0]} == vocab {audit['config_vocab_size']}: "
      f"{audit['lm_head_out_equals_vocab']}")
baseline = generate(model, tokenizer, config.DOMAIN_PROMPTS)
save_json([{"prompt": p, "base_output": o} for p, o in zip(config.DOMAIN_PROMPTS, baseline)],
          config.EVAL_DIR / "baseline_generations.json")
display(pd.DataFrame({"prompt": config.DOMAIN_PROMPTS, "base model output (before CPT)": baseline}))
del model; import gc; gc.collect(); torch.cuda.empty_cache()

**Inferences — Step 3**
* **Parameter count** ≈ 1.10 B, all trainable for CPT (full-parameter continual pre-training, not adapters).
* **Architecture:** 22 decoder layers, hidden size 2048, 32 query heads of dimension 64 (2048 / 32), but only **4 key/value heads** — *grouped-query attention* (8 query heads share one KV head). GQA shrinks the KV-cache 8× and the k/v projection matrices to 2048×256, which is why `k_proj`/`v_proj` are much smaller than `q_proj`/`o_proj` in the printout; SwiGLU MLP with intermediate size 5632; RoPE positions up to 2048.
* **lm_head check:** the output projection is 2048 → 32 000 and equals the tokenizer/config vocabulary, so every logit maps to exactly one token id — the precondition for computing cross-entropy and perplexity correctly. Embeddings are *not* tied in TinyLlama.
* **Baseline:** the base model produces fluent, generic English but invents numbers and does not know the MyGov specifics (probation leave = casual/sick only, Rs.10/km own-vehicle claim, 60-day written notice). These outputs are the reference point for Step 5 and Part B.
* bf16 is *emulated* on a T4 (compute capability 7.5); it works for loading/inference, but the training step below uses fp32 master weights with fp16 autocast (see Step 4).

## Step 4 — CPT training loop & loss analysis  [2 marks]

* `PackedDataset` (a `torch.utils.data.Dataset`) wraps the Parquet blocks; `labels = input_ids` (the causal shift happens inside the model).
* `LossRecorderCallback` (a custom `TrainerCallback`) captures loss, learning rate and grad-norm at **every** logging step (`logging_steps=1`) and the held-out loss every ~1/8 of training.

| Hyper-parameter | Value | Reason |
|---|---|---|
| learning rate | 2e-5, cosine, 10 % warm-up | ~10× below TinyLlama's pre-training peak LR; small enough to limit catastrophic forgetting |
| effective batch | 1 × 8 grad-accum × 2048 = 16 384 tokens/update | fits T4 memory, stable gradients |
| epochs | 2 | small corpus — a second pass helps adaptation; more risks memorisation & forgetting |
| precision | fp32 master weights + fp16 autocast (T4) / bf16 autocast (A100) | pure-bf16 Adam updates of ~2e-5 fall below bf16 resolution and are lost |
| optimiser | paged 8-bit AdamW | 1.1 B × 2 Adam states in 8-bit ≈ 2.2 GB instead of 8.8 GB |
| memory savers | gradient checkpointing, batch 1 | activations recomputed in backward |
| grad clip / weight decay | 1.0 / 0.01 | standard |

*If the T4 runs out of memory, set `os.environ["BLOCK_SIZE"]="1024"` and re-run from Step 2.*

In [ ]:
from src.train_cpt import train_cpt
cpt_summary, loss_rows = train_cpt()
display(pd.Series(cpt_summary).to_frame("value"))

In [ ]:
from src.evaluate_cpt import plot_loss_curve
loss_stats, png = plot_loss_curve()
display(Image(str(png)))
display(pd.Series(loss_stats).to_frame("value"))
print(f"Starting loss (pretrained, measured before step 1): {cpt_summary['start_train_loss']:.3f} "
      f"→ final logged loss {loss_stats['last_logged_loss']:.3f} ({loss_stats['loss_drop_percent']}% lower); "
      f"plateau ≈ step {loss_stats['plateau_step']}")

**Inferences — Step 4**
* **Starting point:** the loss before the first update is ≈ 2–3 (printed above) — consistent with a properly loaded pretrained model (a randomly initialised LLaMA would start near ln 32 000 ≈ 10.4; `train_cpt` raises an error if the start loss is > 8).
* **Shape of the curve:** a fast drop during warm-up and the first ~25 % of steps as the model picks up the domain's recurring vocabulary and formatting (band tables, “Rs. …/-”, “shall be”, numbered clauses), then a slow decline and a **plateau** (dashed line) where it is mostly memorising document-specific detail. The small step-to-step jitter is expected with batch size 1 × 8 and very heterogeneous documents.
* **Train vs. held-out loss:** held-out loss falls with training loss in the first epoch. If it flattens or rises in epoch 2 while training loss keeps falling, the model has started to over-fit the training policies — the signal to stop at ~1 epoch or lower the LR.
* **Checkpoint:** the final CPT model + tokenizer are saved to `models/cpt/` on Drive (bf16 safetensors, ~2.2 GB) for Step 5 and Part B.

## Step 5 — Evaluation: perplexity & catastrophic forgetting  [2 marks]

### 5A — Domain perplexity
$$\text{PPL} = \exp\Big(-\tfrac{1}{N}\sum_{i}\log P(t_i\mid t_1\dots t_{i-1})\Big)$$
computed with a no-grad cross-entropy loop over **every** token of the held-out documents (never seen in CPT), for the base and the CPT model.

In [ ]:
from src.evaluate_cpt import domain_perplexity
ppl = domain_perplexity()
display(pd.Series(ppl).to_frame("value"))
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["Base", "CPT"], [ppl["base_ppl"], ppl["cpt_ppl"]], color=["#9db4d6", "#1f4e8c"])
for i, v in enumerate([ppl["base_ppl"], ppl["cpt_ppl"]]): ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
ax.set_title(f"Held-out domain perplexity (−{ppl['ppl_reduction_percent']}%)"); plt.tight_layout()
plt.savefig(config.FIGURE_DIR / "perplexity_base_vs_cpt.png", dpi=150); plt.show()

### 5B — Catastrophic-forgetting check
Both models complete three general-knowledge prompts (greedy decoding). Verdict: **Retained** if the CPT output still contains the fact, **Degraded** if the base model had it and the CPT model lost it.

In [ ]:
from src.evaluate_cpt import forgetting_check
forgetting, domain_gen = forgetting_check()
display(forgetting[["prompt", "base_output", "cpt_output", "expected_fact", "verdict"]])
print("Domain prompts — base vs CPT completions:")
display(domain_gen)

**Inferences — Step 5**
* **Domain PPL:** the CPT model's perplexity on *unseen* HR policies is lower than the base model's (reduction printed above; the assignment's typical range is 10–40 %). Because the held-out documents were never trained on, the gain is not memorisation — the model has learned the statistical regularities of HR-policy language (clause numbering, entitlement tables, legal register, Indian-English terms such as *resource*, *band*, *notice period*, *ICC*).
* If the reduction is above ~40 %, check that near-duplicates were removed (Step 1) — a duplicate across the split would inflate the gain; if it is below 10 %, the corpus/epochs were too small for the LR.
* **Forgetting:** with LR 2e-5 and only 2 epochs over a small corpus, the weight change is small, so general facts (Paris, 100 °C, ~300 000 km/s) are expected to be **Retained**; any wording change is stylistic (the CPT model tends to continue in a formal, policy-like register). Per the assignment tip, if a fact were *Degraded*, the remedy is to cut the LR 10× or halve the steps.
* **Domain completions:** compare the two columns — after CPT the completions adopt the policy style and are more likely to state the correct entitlement, but a 1.1 B base model given a raw prompt still hallucinates details; CPT teaches *language*, not reliable *question answering*, which motivates Part B.

# PART B — Instruction Fine-Tuning with QLoRA

## B1 — Instruction dataset creation  [2 marks]
**Source:** the cleaned `data/domain_corpus/*.txt` from Step 1. **Format:** JSONL with `instruction` and `response` (+ provenance: `source`, `section`, `type`).

**Method — heuristic generation (`src/instruction_data.py`, default; offline & reproducible):** each policy is parsed into numbered/ALL-CAPS sections and list-item clauses, then converted with four templates; every response is copied verbatim from the policy, so answers are grounded by construction:

| Type | Instruction template | Response |
|---|---|---|
| `section_qa` | What does the *{policy}* say about *{section}*? | section text (≤ 180 words) |
| `section_sum` | Summarise the *'{section}'* provisions of the *{policy}*. | first two sentences |
| `clause_rule` | What does the *{policy}* require regarding *{subject}*? (subject = noun phrase before *shall / must / is entitled / is eligible …*) | the clause |
| `clause_cloze` | Complete this rule from the *{policy}* (section …): "*first half* …" | the full clause |

Long manuals are capped at 150 pairs per document so they do not dominate. Duplicated instructions are removed.

**Optional — LLM synthetic generation** (`build_instruction_dataset("llm")`, needs `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`). Exact prompt template used per 600-word chunk:
```
Read the text below and generate {n} instruction-response pairs in JSON format based ONLY on this text. Each entry must have instruction and response keys.
The instruction must be a question an employee or HR executive would realistically ask; the response must answer it using only facts stated in the text (quote numbers, limits and conditions exactly). Return a JSON list only, with no commentary.

Document: {title}
---
{chunk}
---
```
**Split:** 80 / 20 by *section group* — all pairs generated from the same section go to the same split, so the eval set measures generalisation to unseen clauses rather than re-phrasings of training clauses.

In [ ]:
from src.instruction_data import build_instruction_dataset
METHOD = "heuristic"   # or "llm" if an API key is configured
inst_stats = build_instruction_dataset(METHOD)
inst = pd.read_json(config.INSTRUCTION_DIR / "instruction_dataset.jsonl", lines=True)
print(f"Train: {inst_stats['train_pairs']} | Eval: {inst_stats['eval_pairs']} | Total: {inst_stats['total_pairs']}")
display(inst.groupby("type").size().to_frame("pairs"))
display(inst.sample(8, random_state=1)[["type", "instruction", "response", "source"]])

**Inferences — B1**
* A ~1.1 B model needs on the order of 10³ examples to learn the *format* “answer the question from the policy” — the dataset size above is in that range and every answer is traceable to a source file and section.
* Heuristic pairs are 100 % faithful to the text (no hallucinated answers) but their questions are templated and some section answers are long; an LLM-generated set would have more natural, varied questions at the cost of needing a factuality spot-check. The template is kept in this notebook for that option.
* Section-grouped splitting gives a slightly uneven 80/20 (printed above) but avoids train/eval leakage.

## B2 — QLoRA fine-tuning with **Adapter B (Balanced)**  [2 marks]
| Setting | Value |
|---|---|
| Base weights | CPT checkpoint from Step 4, **4-bit NF4**, double quantisation, compute dtype fp16 (T4) |
| Adapter | **B — r = 16, α = 32, target `q_proj`, `v_proj`**, dropout 0.05 |
| Trainer | TRL `SFTTrainer`, prompt/completion conversations rendered with the chat template; loss on the assistant completion only |
| Optimiser / LR | paged 8-bit AdamW, 2e-4, cosine, 5 % warm-up, 2 epochs, batch 4 × grad-accum 4, max length 512 |

**Why Adapter B?** Adapter A (r = 8) risks under-fitting multi-clause answers; Adapter C (r = 32 + `o_proj`) roughly doubles adapter parameters and VRAM for little gain on a ~10³-pair dataset. B is the quality/cost balance point. α/r = 2 keeps the effective LoRA scale the same as the common r=8/α=16 setting.

**Chat template:** the TinyLlama *base* tokenizer has no chat template, so the official TinyLlama-1.1B-Chat-v1.0 (Zephyr-style) template — `<|system|> … </s> <|user|> … </s> <|assistant|> …` — is attached to the same tokenizer (no vocabulary change: the markers are ordinary text tokens).

In [ ]:
from src.train_qlora import train_qlora, load_chat_tokenizer
chat_tok = load_chat_tokenizer()
print(chat_tok.apply_chat_template([{"role": "system", "content": config.SYSTEM_PROMPT},
                                    {"role": "user", "content": "How many days of notice are required?"},
                                    {"role": "assistant", "content": "60 days."}], tokenize=False))
sft_summary, sft_rows = train_qlora("B")

In [ ]:
log = pd.DataFrame(sft_rows)
fig, ax = plt.subplots(figsize=(8, 3.8))
t = log.dropna(subset=["loss"]); e = log.dropna(subset=["eval_loss"])
ax.plot(t.step, t.loss, label="train loss"); ax.plot(e.step, e.eval_loss, "o-", label="eval loss")
ax.set_xlabel("step"); ax.set_ylabel("loss (completion tokens)"); ax.set_title("QLoRA Adapter B"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(config.FIGURE_DIR / "sft_adapter_B_loss.png", dpi=150); plt.show()
display(pd.Series({k: v for k, v in sft_summary.items() if k != "hyperparameters"}).to_frame("value"))

**Inferences — B2**
* Only ~0.2 % of parameters (the LoRA A/B matrices on `q_proj`/`v_proj`; ≈2.3 M) are trained; the 4-bit base needs < 1 GB, so peak VRAM is a few GB — far below the full-parameter CPT run.
* Completion-only loss starts relatively high (the model has never been asked questions in this format) and drops quickly in the first epoch as it learns the `<|assistant|>` answer format and to end with `</s>`; eval loss tracking train loss means the adapter generalises to unseen sections rather than memorising.

## B3 — Evaluation on the same 3 domain prompts  [1 mark]
The three Step-3/Step-5 topics are asked as questions through the chat template. Columns: raw completions of the base and CPT models (Steps 3/5), the CPT model through the chat template with the adapter **disabled**, and the CPT model **with Adapter B**. Held-out pairs are also scored with ROUGE-L against the reference answers.

In [ ]:
from src.evaluate_sft import evaluate_adapter
domain_cmp, general_cmp, heldout, sft_eval = evaluate_adapter("B")
display(domain_cmp)
display(general_cmp)
display(pd.Series(sft_eval).to_frame("value"))
display(heldout.head(5)[["instruction", "reference", "cpt_no_adapter", "sft_adapter", "rougeL_cpt", "rougeL_sft"]])

**Observations — B3**
* **Format:** without the adapter the CPT model continues the chat text like a document (repeats the question, drifts into other clauses or never stops); with Adapter B it answers directly, in the policy's own wording, and stops with `</s>` — the main effect of instruction tuning.
* **Grounding:** check the `sft_has_fact` column — the adapter answers state the reference entitlement (casual/sick leave on probation, Rs.10 per km, 60-day notice) far more often than the base or CPT model. Where it still misses, it usually quotes a neighbouring clause of the right policy: the knowledge comes from CPT, the *retrieval of the right clause* is the weak point of a 1.1 B model.
* **Held-out ROUGE-L** rises substantially with the adapter, confirming the improvement is not limited to the three hand-picked prompts.
* **General knowledge** after SFT (second table) stays intact — LoRA leaves the base weights frozen.
* **Limitations / next steps:** templated heuristic questions make the model sensitive to phrasing; LLM-generated paraphrases, Adapter C, or retrieval-augmented prompting (Assignment 2B) would improve robustness.

## Summary
| Metric | Result |
|---|---|
| Corpus after cleaning | see Step 1 summary |
| Tokens / packed 2048-blocks | see Step 2 |
| CPT loss (start → end) | see Step 4 |
| Domain PPL base → CPT | see Step 5A |
| Forgetting verdicts | see Step 5B |
| Instruction pairs train / eval | see B1 |
| Adapter B ROUGE-L (no adapter → adapter) | see B3 |

In [ ]:
rows = {
  "documents (raw → clean)": f"{summary['documents_before']} → {summary['documents_after']}",
  "train tokens / blocks": f"{tok_stats['splits']['train']['total_tokens']:,} / {tok_stats['splits']['train']['packed_sequences']}",
  "CPT loss start → end": f"{cpt_summary['start_train_loss']:.3f} → {loss_stats['last_logged_loss']:.3f}",
  "domain PPL base → CPT": f"{ppl['base_ppl']:.2f} → {ppl['cpt_ppl']:.2f} (−{ppl['ppl_reduction_percent']}%)",
  "forgetting verdicts": ", ".join(forgetting.verdict),
  "instruction pairs train / eval": f"{inst_stats['train_pairs']} / {inst_stats['eval_pairs']}",
  "ROUGE-L no adapter → Adapter B": f"{sft_eval['rougeL_cpt_no_adapter']:.3f} → {sft_eval['rougeL_sft_adapter']:.3f}",
}
display(pd.Series(rows).to_frame("result"))

## Export
Saves this notebook as HTML (deliverable) and copies `instruction_dataset.jsonl` to the project root.

In [ ]:
import shutil
shutil.copy(config.INSTRUCTION_DIR / "instruction_dataset.jsonl", "instruction_dataset.jsonl")
# Save the notebook first (File → Save), then:
NB = "notebooks/CorpPolicyLM_Assignment1A.ipynb"
subprocess.run(["jupyter", "nbconvert", "--to", "html", NB, "--output-dir", "notebooks"], check=False)
print(sorted(os.listdir("notebooks")))